In [0]:
try:
    ano_mes_pasta = dbutils.jobs.taskValues.get(taskKey="ingestao_bronze", key="ano_mes_pasta", debugValue='2026_09')
    if not ano_mes_pasta:
        raise ValueError("ano_mes_pasta não retornado pela task 'ingestao_bronze' (taskValues vazio ou key ausente)")

    catalog_schema = 'databricks_cnpj_data_lakehouse.gold'

    df_tables = spark.sql(f'SHOW TABLES IN {catalog_schema}').collect()

    for table in df_tables:
        
        table_name = table.tableName        

        sql_alter_table = f"ALTER TABLE {catalog_schema}.{table_name}"

        match table_name:
            case 'agg_fact_estabelecimentos_cnaes':
                continue

            case 'agg_fact_estabelecimentos':
                continue
            
            case 'bridge_estabelecimentos_socios':
                spark.sql(f"{sql_alter_table} ALTER COLUMN sk_estabelecimento SET NOT NULL")
                spark.sql(f"{sql_alter_table} ALTER COLUMN sk_socio SET NOT NULL")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS pk_{table_name}_sk_id")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT pk_{table_name}_sk_id PRIMARY KEY (sk_estabelecimento, sk_socio)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_bridge_estabelecimentos_socios_fact_estabelecimentos")                             

            case 'bridge_estabelecimentos_cnaes':
                spark.sql(f"{sql_alter_table} ALTER COLUMN sk_estabelecimento SET NOT NULL")
                spark.sql(f"{sql_alter_table} ALTER COLUMN sk_cnae SET NOT NULL")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS pk_{table_name}_sk_id")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT pk_{table_name}_sk_id PRIMARY KEY (sk_estabelecimento, sk_cnae)")
                            
            case 'fact_estabelecimentos':
                spark.sql(f"{sql_alter_table} ALTER COLUMN sk_estabelecimento SET NOT NULL")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS pk_{table_name}_sk_estabelecimento CASCADE")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT pk_{table_name}_sk_estabelecimento PRIMARY KEY (sk_estabelecimento)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_tempo_inicio_atividades") 
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_fact_estabelecimentos_dim_tempo_inicio_atividades FOREIGN KEY (sk_tempo_inicio_atividade) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_tempo_inicio_atividades(sk_id)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_tempo_situacao_cadastral")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_fact_estabelecimentos_dim_tempo_situacao_cadastral FOREIGN KEY (sk_tempo_situacoes_cadastrais) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_tempo_situacoes_cadastrais(sk_id)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_tempo_situacao_especial")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_fact_estabelecimentos_dim_tempo_situacao_especial FOREIGN KEY (sk_tempo_situacoes_especiais) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_tempo_situacoes_especiais(sk_id)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_situacoes")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_fact_estabelecimentos_dim_situacoes FOREIGN KEY (sk_situacoes) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_situacoes(sk_id)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_localidades")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_fact_estabelecimentos_dim_localidades FOREIGN KEY (sk_localidades) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_localidades(sk_id)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_naturezas_juridicas")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_fact_estabelecimentos_dim_naturezas_juridicas FOREIGN KEY (sk_naturezas_juridicas) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_naturezas_juridicas(sk_id)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_fact_estabelecimentos_dim_cadastro")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_fact_estabelecimentos_dim_cadastro FOREIGN KEY (sk_estabelecimento) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_cadastro(sk_id)")    
                spark.sql(f"{sql_alter_table} CLUSTER BY (sk_tempo_inicio_atividade, sk_situacoes, sk_localidades, sk_naturezas_juridicas)")

            case _:    
                spark.sql(f"{sql_alter_table} ALTER COLUMN sk_id SET NOT NULL")                
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS pk_{table_name}_sk_id CASCADE")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT pk_{table_name}_sk_id PRIMARY KEY (sk_id)")                

        spark.sql(f"OPTIMIZE {catalog_schema}.{table_name} FULL")

    for table in df_tables:
        
        table_name = table.tableName        

        sql_alter_table = f"ALTER TABLE {catalog_schema}.{table_name}"

        match table_name:
            case 'agg_fact_estabelecimentos_cnaes':
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_cnaes_dim_cnaes")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_agg_fact_estabelecimentos_cnaes_dim_cnaes FOREIGN KEY (sk_cnae) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_cnaes(sk_id)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_cnaes_dim_tempo_inicio_atividades")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_agg_fact_estabelecimentos_cnaes_dim_tempo_inicio_atividades FOREIGN KEY (sk_tempo_inicio_atividade) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_tempo_inicio_atividades(sk_id)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_cnaes_dim_situacoes")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_agg_fact_estabelecimentos_cnaes_dim_situacoes FOREIGN KEY (sk_situacoes) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_situacoes(sk_id)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_cnaes_dim_localidades")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_agg_fact_estabelecimentos_cnaes_dim_localidades FOREIGN KEY (sk_localidades) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_localidades(sk_id)")                
                spark.sql(f"{sql_alter_table} CLUSTER BY (sk_cnae, sk_localidades, sk_tempo_inicio_atividade, sk_situacoes)")

            case 'agg_fact_estabelecimentos':
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_dim_tempo_inicio_atividades")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_agg_fact_estabelecimentos_dim_tempo_inicio_atividades FOREIGN KEY (sk_tempo_inicio_atividade) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_tempo_inicio_atividades(sk_id)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_dim_situacoes")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_agg_fact_estabelecimentos_dim_situacoes FOREIGN KEY (sk_situacoes) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_situacoes(sk_id)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_agg_fact_estabelecimentos_dim_localidades")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_agg_fact_estabelecimentos_dim_localidades FOREIGN KEY (sk_localidades) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_localidades(sk_id)")
                spark.sql(f"{sql_alter_table} CLUSTER BY (sk_tempo_inicio_atividade, sk_situacoes, sk_localidades)")

            case 'bridge_estabelecimentos_socios':                
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_bridge_estabelecimentos_socios_fact_estabelecimentos") 
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_bridge_estabelecimentos_socios_fact_estabelecimentos FOREIGN KEY (sk_estabelecimento) REFERENCES databricks_cnpj_data_lakehouse.gold.fact_estabelecimentos(sk_estabelecimento)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_bridge_estabelecimentos_socios_dim_socios")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_bridge_estabelecimentos_socios_dim_socios FOREIGN KEY (sk_socio) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_socios(sk_id)")
                spark.sql(f"{sql_alter_table} CLUSTER BY (sk_estabelecimento, sk_socio)") 

            case 'bridge_estabelecimentos_cnaes':
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_bridge_estabelecimentos_cnaes_fact_estabelecimentos")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_bridge_estabelecimentos_cnaes_fact_estabelecimentos FOREIGN KEY (sk_estabelecimento) REFERENCES databricks_cnpj_data_lakehouse.gold.fact_estabelecimentos(sk_estabelecimento)")
                spark.sql(f"{sql_alter_table} DROP CONSTRAINT IF EXISTS fk_bridge_estabelecimentos_cnaes_dim_cnaes")
                spark.sql(f"{sql_alter_table} ADD CONSTRAINT fk_bridge_estabelecimentos_cnaes_dim_cnaes FOREIGN KEY (sk_cnae) REFERENCES databricks_cnpj_data_lakehouse.gold.dim_cnaes(sk_id)")
                spark.sql(f"{sql_alter_table} CLUSTER BY (sk_estabelecimento, sk_cnae)") 

            case 'dim_cadastro':                    
                spark.sql(f"{sql_alter_table} CLUSTER BY (cnpj_completo)")

        spark.sql(f"OPTIMIZE {catalog_schema}.{table_name} FULL")

except Exception as e:
    print(f"FALHA ao processar {e}")
    raise